# TamilScript AI: CRNN Word Recognition Training (Colab T4 GPU)
## Real Handwritten Tamil Word Recognizer (CNN + BiLSTM + CTC Loss)

This notebook trains the deep learning word recognizer on **`c3rl/IIIT-INDIC-HW-WORDS-Tamil`**:
- **Hardware**: Google Colab T4 GPU with Mixed Precision (AMP `fp16`)
- **Architecture**: 7-layer CNN backbone + 2-layer BiLSTM (512 features) + CTC loss projection
- **Preprocessing**: Input height 64 (aspect-ratio preserved), padded width 256, polarity-matched
- **Training**: 30 epochs with Early Stopping (patience 5) on Validation CER
- **Target Gate**: Validation CER < 20% (Target: < 15%)
- **Export**: ONNX format for live browser (<100ms) and FastAPI server inference.

In [ ]:
# 1. GPU Check & Environment Setup
!nvidia-smi
!pip install -q torch torchvision torchaudio datasets pyarrow onnx onnxruntime Levenshtein pillow numpy

In [ ]:
import os
import io
import json
import time
import math
from pathlib import Path
from typing import List, Dict, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

import numpy as np
from PIL import Image, ImageOps, ImageEnhance
import Levenshtein

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'[*] Training on device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"})')

In [ ]:
# 2. Load Dataset from Hugging Face Hub
from datasets import load_dataset

print('[*] Loading c3rl/IIIT-INDIC-HW-WORDS-Tamil...')
dataset = load_dataset('c3rl/IIIT-INDIC-HW-WORDS-Tamil')
print(f'[+] Dataset splits: {dataset.keys()}')

train_split = dataset['train'] if 'train' in dataset else dataset['test'].train_test_split(test_size=0.15, seed=42)['train']
val_split = dataset['test'] if 'train' in dataset else dataset['test'].train_test_split(test_size=0.15, seed=42)['test']
print(f'[+] Train samples: {len(train_split):,}, Validation samples: {len(val_split):,}')

In [ ]:
# 3. Build CTC Vocabulary covering 100% of Tamil Codepoints
all_chars = set()
for sample in train_split:
    if sample.get('text'):
        all_chars.update(sample['text'].strip())
for sample in val_split:
    if sample.get('text'):
        all_chars.update(sample['text'].strip())

sorted_chars = sorted(list(all_chars))
# Index 0 is reserved for CTC blank
ctc_vocab = {'<blank>': 0}
for idx, ch in enumerate(sorted_chars, start=1):
    ctc_vocab[ch] = idx

id_to_char = {idx: ch for ch, idx in ctc_vocab.items()}
print(f'[+] Total CTC Classes: {len(ctc_vocab)} (including blank at index 0)')
with open('tamil_ctc_vocab.json', 'w', encoding='utf-8') as f:
    json.dump(ctc_vocab, f, ensure_ascii=False, indent=2)

In [ ]:
# 4. Preprocessing: Height 64, Aspect-Ratio Preserved, Padded Width 256, Inverted Polarity
TARGET_H = 64
TARGET_W = 256

def preprocess_word_image(pil_img: Image.Image, augment: bool = False) -> torch.Tensor:
    gray = pil_img.convert('L')
    arr = np.array(gray)
    
    # Standardize polarity: black background (0), bright ink (255)
    if np.mean(arr) > 127:
        gray = ImageOps.invert(gray)
        arr = np.array(gray)
        
    # Crop ink bounding box
    y_idx, x_idx = np.where(arr > 30)
    if len(y_idx) > 0 and len(x_idx) > 0:
        crop = gray.crop((int(np.min(x_idx)), int(np.min(y_idx)), int(np.max(x_idx)) + 1, int(np.max(y_idx)) + 1))
    else:
        crop = gray
        
    if augment:
        # Slight rotation (-4 to +4 degrees)
        angle = np.random.uniform(-4.0, 4.0)
        crop = crop.rotate(angle, resample=Image.Resampling.BILINEAR, fillcolor=0)
        # Contrast jitter
        enhancer = ImageEnhance.Contrast(crop)
        crop = enhancer.enhance(np.random.uniform(0.8, 1.2))
        
    w, h = crop.size
    if h == 0 or w == 0:
        return torch.zeros(1, TARGET_H, TARGET_W, dtype=torch.float32)
        
    # Scale to height 64 preserving aspect ratio
    new_w = min(TARGET_W, max(12, int(w * (TARGET_H / h))))
    resized = crop.resize((new_w, TARGET_H), Image.Resampling.BILINEAR)
    
    # Pad into TARGET_W
    padded = Image.new('L', (TARGET_W, TARGET_H), color=0)
    padded.paste(resized, (0, 0))
    
    norm = np.array(padded, dtype=np.float32) / 255.0
    return torch.tensor(norm, dtype=torch.float32).unsqueeze(0)

In [ ]:
# 5. PyTorch Dataset & Collate Function
class TamilWordDataset(Dataset):
    def __init__(self, split_data, vocab: Dict[str, int], augment: bool = False):
        self.vocab = vocab
        self.augment = augment
        self.samples = []
        for item in split_data:
            text = item.get('text', '').strip() if item.get('text') else ''
            img = item.get('image')
            if text and img:
                if all(c in self.vocab for c in text):
                    self.samples.append((text, img))
        print(f'[+] Loaded {len(self.samples):,} valid samples (augment={augment})')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        text, img = self.samples[idx]
        if not isinstance(img, Image.Image):
            img = Image.open(io.BytesIO(img['bytes']) if isinstance(img, dict) else img)
        tensor = preprocess_word_image(img, augment=self.augment)
        targets = [self.vocab[c] for c in text]
        return tensor, torch.tensor(targets, dtype=torch.long), text

def collate_fn(batch):
    tensors, targets, texts = zip(*batch)
    batch_tensors = torch.stack(tensors, dim=0) # (B, 1, 64, 256)
    target_lengths = torch.tensor([len(t) for t in targets], dtype=torch.long)
    flat_targets = torch.cat(targets)
    return batch_tensors, flat_targets, target_lengths, texts

train_ds = TamilWordDataset(train_split, ctc_vocab, augment=True)
val_ds = TamilWordDataset(val_split, ctc_vocab, augment=False)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, collate_fn=collate_fn, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, collate_fn=collate_fn, num_workers=2, pin_memory=True)

In [ ]:
# 6. Deep CRNN Model Definition (H=64 -> 1, BiLSTM hidden=256)
class TamilDeepCRNN(nn.Module):
    def __init__(self, num_classes: int = 65, hidden_size: int = 256):
        super().__init__()
        # CNN Backbone: 64 -> 32 -> 16 -> 8 -> 4 -> 2 -> 1
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(True), nn.MaxPool2d(2, 2), # 32, W/2
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(True), nn.MaxPool2d(2, 2), # 16, W/4
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(True),
            nn.Conv2d(256, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(True), nn.MaxPool2d((2, 1), (2, 1)), # 8, W/4
            nn.Conv2d(256, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(True), nn.MaxPool2d((2, 1), (2, 1)), # 4, W/4
            nn.Conv2d(512, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(True), nn.MaxPool2d((2, 1), (2, 1)), # 2, W/4
            nn.Conv2d(512, 512, 2, padding=0), nn.BatchNorm2d(512), nn.ReLU(True) # 1, W/4 - 1
        )
        self.rnn = nn.LSTM(512, hidden_size, bidirectional=True, num_layers=2, batch_first=True, dropout=0.25)
        self.fc = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        feat = self.cnn(x) # (B, 512, 1, T)
        seq = feat.squeeze(2).permute(0, 2, 1) # (B, T, 512)
        out, _ = self.rnn(seq)
        logits = self.fc(out) # (B, T, num_classes)
        return logits.permute(1, 0, 2) # (T, B, num_classes) for CTCLoss

    def decode_greedy(self, logits: torch.Tensor) -> List[List[int]]:
        preds = torch.argmax(logits, dim=-1).permute(1, 0).cpu().numpy()
        results = []
        for seq in preds:
            res, prev = [], -1
            for t in seq:
                if t != prev:
                    if t != 0: res.append(int(t))
                    prev = t
            results.append(res)
        return results

model = TamilDeepCRNN(num_classes=len(ctc_vocab), hidden_size=256).to(device)
print(f'[+] Model parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# 7. Training Loop: 30 Epochs with Early Stopping on Validation CER
EPOCHS = 30
PATIENCE = 5
ctc_loss = nn.CTCLoss(blank=0, zero_infinity=True)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scaler = GradScaler()
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

best_cer = float('inf')
patience_counter = 0

print('[*] Commencing 30-Epoch CRNN Training on T4 GPU...')
print(f'{"Epoch":<7} | {"Loss":<8} | {"Val CER":<10} | {"Val WER":<10} | {"Time":<6} | {"Status"}')
print('-' * 60)

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    t0 = time.time()
    
    for bx, flat_targets, target_lengths, _ in train_loader:
        bx = bx.to(device, non_blocking=True)
        flat_targets = flat_targets.to(device, non_blocking=True)
        b_size = bx.size(0)
        
        optimizer.zero_grad()
        with autocast():
            logits = model(bx)
            input_lengths = torch.full((b_size,), logits.size(0), dtype=torch.long, device=device)
            log_probs = logits.log_softmax(2)
            loss = ctc_loss(log_probs, flat_targets, input_lengths, target_lengths)
            
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item()
        
    scheduler.step()
    avg_loss = train_loss / len(train_loader)
    
    # Validation Evaluation
    model.eval()
    char_edits, ref_chars, word_errors, total_words = 0, 0, 0, 0
    with torch.no_grad():
        for bx, _, _, texts in val_loader:
            bx = bx.to(device, non_blocking=True)
            logits = model(bx)
            decoded_seqs = model.decode_greedy(logits)
            for true_text, pred_ids in zip(texts, decoded_seqs):
                pred_text = ''.join([id_to_char.get(i, '') for i in pred_ids if i != 0])
                char_edits += Levenshtein.distance(true_text, pred_text)
                ref_chars += len(true_text)
                if true_text != pred_text:
                    word_errors += 1
                total_words += 1
                
    cer = (char_edits / max(1, ref_chars)) * 100.0
    wer = (word_errors / max(1, total_words)) * 100.0
    elapsed = time.time() - t0
    
    status = ''
    if cer < best_cer:
        best_cer = cer
        patience_counter = 0
        torch.save(model.state_dict(), 'crnn_best.pt')
        status = f'⭐ Best (CER {cer:.2f}%)'
    else:
        patience_counter += 1
        
    print(f'{epoch:<7} | {avg_loss:<8.4f} | {cer:<10.2f}% | {wer:<10.2f}% | {elapsed:<5.1f}s | {status}')
    
    if patience_counter >= PATIENCE:
        print(f'[*] Early stopping triggered after {epoch} epochs (Best CER: {best_cer:.2f}%).')
        break

print(f'\n[+] Training Complete! Best Validation CER: {best_cer:.2f}%')

In [ ]:
# 8. Final Evaluation & 20 Sample Test Print
model.load_state_dict(torch.load('crnn_best.pt', map_location=device))
model.eval()

print('=' * 80)
print('FINAL MODEL EVALUATION ON 20 TEST SAMPLES')
print('=' * 80)
for i in range(20):
    tensor, _, true_text = val_ds[i]
    with torch.no_grad():
        logits = model(tensor.unsqueeze(0).to(device))
        pred_ids = model.decode_greedy(logits)[0]
        pred_text = ''.join([id_to_char.get(tok, '') for tok in pred_ids if tok != 0])
    ed = Levenshtein.distance(true_text, pred_text)
    match = 'MATCH' if true_text == pred_text else 'ERROR'
    print(f'[{match:5s}] #{i+1:2d} | True: {true_text:<16} | Pred: {pred_text:<16} | Edit Dist: {ed}')

In [ ]:
# 9. Export to ONNX (Dynamic Width)
dummy_input = torch.randn(1, 1, 64, 256, device=device)
onnx_path = 'crnn.onnx'
torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    input_names=['input'],
    output_names=['logits'],
    dynamic_axes={'input': {0: 'batch', 3: 'width'}, 'logits': {0: 'time', 1: 'batch'}},
    opset_version=14
)
print(f'[+] Exported CRNN to ONNX: {onnx_path} ({os.path.getsize(onnx_path) / 1024:.1f} KB)')

# Download instructions for local app integration
from google.colab import files
print('[*] Downloading crnn.onnx and crnn_best.pt to replace local checkpoints...')
files.download('crnn.onnx')
files.download('crnn_best.pt')